# 🧬 HAPLO-BENCH: Générateur d'instances de benchmark pour l'inférence d'haplotypes

Ce notebook Google Colab interactif vous permet d'installer le package `haplo-bench` directement depuis votre dépôt GitHub, de configurer graphiquement vos paramètres de simulation biologiques et physiques (grâce aux formulaires Colab), et de générer un ensemble de graphes signés pondérés avec des graines aléatoires (seeds) consécutives de **$1$ à $N$**.

---

In [ ]:
#@title 🛠️ Installation de HAPLO-BENCH
#@description Installe le package haplo-bench et clone le dépôt Haplotypes.
import os

print("📥 Clonage du dépôt Haplotypes depuis GitHub...")
!git clone https://github.com/Ludwig-H/Haplotypes.git

%cd Haplotypes

print("⚙️ Installation de la bibliothèque haplo-bench en mode éditable...")
!pip install -e .

print("✅ Installation terminée !")

## ⚙️ Configuration du Générateur

Remplissez le formulaire ci-dessous pour configurer vos paramètres de simulation.

*   **preset** : Le profil technologique de séquençage à émuler.
*   **chromosome** : Le chromosome d'intérêt (preset de longueurs).
*   **density** ($\rho_{\text{het}}$) : La densité de variants hétérozygotes.
*   **coverage** : La couverture cible du séquençage.
*   **min_shared_variants** : Nombre minimum de variants requis pour connecter deux reads.
*   **enable_triangles** : Activer le calcul d'ordre supérieur (triangles signés).
*   **triangle_rule** : Règle de gel de triangles.
*   **num_graphs** : Nombre de graphes à générer (avec seed allant de $1$ à `num_graphs`).

In [ ]:
#@title 🧬 Formulaire de Paramètres de Simulation { display-mode: "form" }

preset = "pacbio_hifi" #@param ["theory_fixed", "illumina_pe150", "pacbio_hifi", "ont_q20", "ont_duplex", "ont_ultralong"]
chromosome = "chr22" #@param ["chr20", "chr22", "chr1", "chr2"]
density = 0.00075 #@param {type:"number"}
coverage = 30 #@param {type:"integer"}
min_shared_variants = 1 #@param {type:"integer"}
enable_triangles = True #@param {type:"boolean"}
triangle_rule = "edge_clique" #@param ["edge_clique", "shared_variant", "shared_region"]
num_graphs = 5 #@param {type:"integer"}
output_dir = "data/benchmark" #@param {type:"string"}

print("Paramètres enregistrés !")

## 🚀 Génération des instances de graphe en boucle

La cellule ci-dessous va générer les dossiers d'instances correspondants.
Par défaut, la graine aléatoire (**seed**) de génération va automatiquement varier de **$1$ au nombre de graphes demandés** (de $1$ à `{num_graphs}`).

Pour chaque étape, un fichier YAML temporaire est généré et passé à la commande `haplo-bench generate`.

In [ ]:
import yaml
import os
import subprocess

# Assurer la création du dossier parent
os.makedirs(output_dir, exist_ok=True)

# Création de la configuration modèle
base_config = {
    "preset": preset,
    "reference": {
        "assembly": "GRCh38.p14",
        "chromosome": chromosome
    },
    "variants": {
        "model": "bernoulli",
        "density": density
    },
    "coverage": {
        "target": coverage,
        "compute_n_reads": True
    },
    "graph": {
        "min_shared_variants": min_shared_variants,
        "edge_rule": "likelihood_ratio"
    },
    "higher_order": {
        "enable_triangles": enable_triangles,
        "triangle_rule": triangle_rule
    }
}

print(f"🚀 Début de la génération de {num_graphs} instances...")

for seed in range(1, num_graphs + 1):
    print(f"\n" + "="*60)
    print(f"▶️ Instance {seed} / {num_graphs} (Random Seed: {seed})")
    print("="*60)
    
    # Config spécifique à l'instance avec la seed
    config = base_config.copy()
    config["seed"] = seed
    
    # Écrire le fichier de configuration temporaire
    temp_yaml = f"temp_config_seed_{seed}.yaml"
    with open(temp_yaml, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False)
        
    instance_path = os.path.join(output_dir, f"instance_seed_{seed}")
    
    # Construction de la commande CLI
    cmd = [
        "haplo-bench", "generate",
        "--config", temp_yaml,
        "--out", instance_path,
        "--seed", str(seed)
    ]
    
    print(f"Exécution : {' '.join(cmd)}")
    
    try:
        # Exécution
        res = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(res.stdout)
        print(f"✅ Instance générée avec succès dans : {instance_path}")
    except subprocess.CalledProcessError as e:
        # Essai de fallback sans l'argument CLI --seed (au cas où l'outil n'accepte la seed que dans le YAML)
        print(f"⚠️ Échec CLI direct : {e.stderr.strip()}")
        print("🔄 Tentative de fallback via la seed contenue dans le fichier YAML...")
        
        cmd_fallback = [
            "haplo-bench", "generate",
            "--config", temp_yaml,
            "--out", instance_path
        ]
        try:
            res_fb = subprocess.run(cmd_fallback, capture_output=True, text=True, check=True)
            print(res_fb.stdout)
            print(f"✅ Instance générée avec succès (via YAML seed) dans : {instance_path}")
        except subprocess.CalledProcessError as e_fb:
            print(f"❌ Échec définitif de génération : {e_fb.stderr.strip()}")
            
    # Supprimer le fichier de configuration temporaire
    if os.path.exists(temp_yaml):
        os.remove(temp_yaml)

print("\n🎉 Génération des graphes terminée !")

## 📊 Visualisation & Analyse des arêtes du graphe signé généré

Exécutez la cellule ci-dessous pour charger les arêtes du premier graphe généré (Seed 1) à l'aide de la bibliothèque Pandas et inspecter le rapport d'instance JSON.

In [ ]:
import pandas as pd
import json
import glob

# Trouver les dossiers générés
instances = sorted(glob.glob(os.path.join(output_dir, "instance_seed_*")))

if instances:
    first_inst = instances[0]
    print(f"🔍 Analyse de l'instance : {first_inst}")
    
    # 1. Lire le rapport sommaire
    summary_json = os.path.join(first_inst, "report", "summary.json")
    if os.path.exists(summary_json):
        with open(summary_json, "r", encoding="utf-8") as f:
            metrics = json.load(f)
        print("\n📈 Métriques clés de l'instance (JSON) :")
        print(json.dumps(metrics, indent=4))
        
    # 2. Charger les arêtes signées et pondérées
    edges_tsv = os.path.join(first_inst, "graph", "edges.tsv")
    if os.path.exists(edges_tsv):
        df = pd.read_csv(edges_tsv, sep="\t")
        print(f"\n📊 Aperçu des premières arêtes du graphe signé pondéré (taille : {len(df)} arêtes) :")
        display(df.head())
else:
    print("❌ Aucune instance générée trouvée dans le répertoire de sortie.")